<a href="https://colab.research.google.com/github/prathamesh-kandpal/FERIA/blob/main/FERIA_Code_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FERIA
### Financial Email Resolution & Inquiry Agent

**Team:** Nexus AI

**Team Members:** Prathamesh, Prajay, Srishti, Mansi, Shefali, Gianna, Priyanshi, Priyanka

## Objective
Build an agentic AI system that:

1. Accepts customer emails
2. Classifies customer intent
3. Redacts sensitive information
4. Retrieves relevant policy context using RAG
5. Drafts compliant responses
6. Escalates high-risk cases to human agents



# System Architecture

## High-Level Flow

Customer Email
   
↓

Triage Agent

↓

PII Redaction Agent

↓

RAG Retrieval Agent

↓

Response Drafting Agent

↓

Final Response / Escalation

---

## Components

### Backend
- FastAPI
- CrewAI

### RAG
- LangChain
- FAISS / ChromaDB

### LLM
- Claude / OpenAI

### Frontend
- HTML
- CSS
- JavaScript

###1. Core Imports

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Dict, Any, List
from pydantic import BaseModel, Field

# LLM (placeholder - configure as needed)
# from langchain.chat_models import ChatOpenAI / Claude wrapper

###2. Define Graph State

In [ ]:
class FERIAState(TypedDict):
    """
    Shared state flowing through the FERIA pipeline.

    This state captures the full lifecycle of an insurance support email,
    from ingestion to resolution or escalation.
    """

    # Raw + processed input
    raw_email: str
    customer_name: Optional[str]
    email_subject: Optional[str]
    redacted_email: Optional[str]

    # PII tracking
    pii_entities: Optional[List[str]]

    # Triage outputs
    insurance_line: Optional[str]
    query_type: Optional[str]
    urgency: Optional[str]
    risk_flags: Optional[List[str]]
    escalation_required: Optional[bool]
    sla_tag: Optional[str]

    # Retrieval outputs
    retrieved_documents: Optional[List[Dict[str, Any]]]

    # Resolution
    draft_response: Optional[str]
    confidence_score: Optional[float]

    # Escalation
    escalation_context: Optional[Dict[str, Any]]

    # Audit
    audit_log: Optional[Dict[str, Any]]

### 3. Define Structured Schemas (Pydantic)

Triage Output

In [ ]:
class TriageOutput(BaseModel):
    """
    Structured classification output for the triage agent.
    """

    insurance_line: str = Field(..., description="Health, Motor, Life, Travel, Home")
    query_type: str = Field(..., description="Type of customer query")
    urgency: str = Field(..., description="Critical, High, Medium, Low")
    risk_flags: List[str] = Field(default_factory=list)
    escalation_required: bool
    sla_tag: str

Retrieval Output

In [ ]:
class RetrievalOutput(BaseModel):
    """
    Output schema for retrieved policy documents.
    """

    documents: List[Dict[str, Any]] = Field(
        ..., description="Relevant policy clauses with metadata"
    )

Resolution Output

In [ ]:
class ResolutionOutput(BaseModel):
    """
    Output schema for generated response.
    """

    response: str
    confidence_score: float

###4. Initialize LLM

In [ ]:
# Placeholder LLM initialization
# Replace with Claude / GPT / other model

llm = None

###5. Build Nodes

5.1 Email Ingestion Node

In [ ]:
def email_ingestion_node(state: FERIAState) -> FERIAState:
    """
    Ingests raw email input into the pipeline.

    Expected Responsibilities:
    - Accept incoming email text
    - Normalize formatting if required
    - Initialize audit trail entry
    """
    pass

5.2 PII Redaction Node

In [ ]:
def pii_redaction_node(state: FERIAState) -> FERIAState:
    """
    Redacts sensitive information from the email.

    Expected Responsibilities:
    - Detect PII (policy numbers, Aadhaar, PAN, etc.)
    - Replace with placeholders
    - Store redacted entity types in audit log
    - Output redacted_email
    """
    pass

5.3 Triage Node

In [ ]:
def triage_node(state: FERIAState) -> FERIAState:
    """
    Classifies the email into structured triage categories.

    Expected Responsibilities:
    - Identify insurance line
    - Detect query type
    - Assign urgency
    - Detect risk flags
    - Decide escalation requirement
    - Assign SLA tag
    """
    pass

5.4 Retrieval (RAG) Node

In [ ]:
def retrieval_node(state: FERIAState) -> FERIAState:
    """
    Retrieves relevant policy documents based on triage output.

    Expected Responsibilities:
    - Query vector database using insurance_line + query_type
    - Retrieve relevant clauses
    - Attach metadata (policy name, section, clause)
    """
    pass

5.5 Resolution Node

In [ ]:
def resolution_node(state: FERIAState) -> FERIAState:
    """
    Generates a customer-facing response for non-escalated cases.

    Expected Responsibilities:
    - Use retrieved documents as grounding
    - Generate clear, empathetic response
    - Include citations
    - Assign confidence score
    """
    pass

5.6 Escalation Handler Node

In [ ]:
def escalation_node(state: FERIAState) -> FERIAState:
    """
    Prepares escalation context for human-in-the-loop handling.

    Expected Responsibilities:
    - Compile full case summary
    - Attach triage outputs
    - Attach retrieved documents
    - Include draft response (if any)
    - Prepare assistant-ready context
    """
    pass

5.7 Audit Node

In [ ]:
def audit_node(state: FERIAState) -> FERIAState:
    """
    Finalizes and stores audit trail for the case.

    Expected Responsibilities:
    - Log PII types redacted
    - Store triage classification
    - Store retrieved document references
    - Record escalation decision
    - Capture SLA + timestamps
    """
    pass

###6. Routing Function

In [ ]:
def route_after_triage(state: FERIAState) -> str:
    """
    Determines whether to proceed to auto-resolution or escalation.

    Routing Logic:
    - If escalation_required is True → escalation_node
    - Else → retrieval_node
    """
    if state.get("escalation_required"):
        return "escalation_node"
    return "retrieval_node"

Post-Resolution Routing

In [ ]:
def route_after_resolution(state: FERIAState) -> str:
    """
    Determines if low-confidence responses should be escalated.

    Routing Logic:
    - If confidence_score is below threshold → escalation_node
    - Else → audit_node
    """
    confidence = state.get("confidence_score", 0)

    if confidence < 0.7:
        return "escalation_node"

    return "audit_node"

###7. Build & Compile Graph

In [ ]:
def build_feria_graph():
    """
    Constructs and compiles the FERIA LangGraph workflow.

    Flow:
    START
      → ingestion
      → pii_redaction
      → triage
      → (conditional)
          → retrieval → resolution → (conditional) → audit / escalation
          → escalation
      → audit
      → END
    """

    builder = StateGraph(FERIAState)

    # Add nodes
    builder.add_node("email_ingestion", email_ingestion_node)
    builder.add_node("pii_redaction", pii_redaction_node)
    builder.add_node("triage", triage_node)
    builder.add_node("retrieval_node", retrieval_node)
    builder.add_node("resolution_node", resolution_node)
    builder.add_node("escalation_node", escalation_node)
    builder.add_node("audit_node", audit_node)

    # Linear flow
    builder.add_edge(START, "email_ingestion")
    builder.add_edge("email_ingestion", "pii_redaction")
    builder.add_edge("pii_redaction", "triage")

    # Conditional routing after triage
    builder.add_conditional_edges(
        "triage",
        route_after_triage,
        {
            "retrieval_node": "retrieval_node",
            "escalation_node": "escalation_node",
        },
    )

    # Retrieval → Resolution
    builder.add_edge("retrieval_node", "resolution_node")

    # Conditional after resolution
    builder.add_conditional_edges(
        "resolution_node",
        route_after_resolution,
        {
            "audit_node": "audit_node",
            "escalation_node": "escalation_node",
        },
    )

    # Escalation → Audit
    builder.add_edge("escalation_node", "audit_node")

    # End
    builder.add_edge("audit_node", END)

    return builder.compile()

###8. Visualize the Graph

In [ ]:
from IPython.display import Image, display

graph = build_feria_graph()

display(Image(graph.get_graph().draw_mermaid_png()))